# Comprobación inicial del entorno

Solo prueba técnica. Los valores siguientes son demostrativos; no representan datos sísmicos reales ni decisiones metodológicas.

## 1. Confirmar intérprete

La comprobación elige la ruta del ejecutable según el sistema operativo y verifica el prefijo de la .venv de este clon.

In [ ]:
import sys
from pathlib import Path

from seismic_wavelets.environment import require_project_python

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "pyproject.toml").is_file()
require_project_python(root, sys.executable, sys.prefix)
print(sys.executable)
print(sys.version)

## 2. Importar las dependencias

Se distingue el nombre de distribución (lo que fija el bloqueo) del módulo Python que se importa. La última ruta confirma de dónde se cargó el paquete del proyecto.

In [ ]:
from importlib import import_module
from importlib.metadata import version

import seismic_wavelets

for distribution, module in [
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("pandas", "pandas"),
    ("PyWavelets", "pywt"),
    ("obspy", "obspy"),
    ("jupyterlab", "jupyterlab"),
    ("ipykernel", "ipykernel"),
    ("pytest", "pytest"),
    ("ruff", "ruff"),
]:
    import_module(module)
    print(f"{distribution}: {version(distribution)}")
print("Paquete local:", seismic_wavelets.__file__)

## 3. Comprobar una DWT reversible

La semilla fija una entrada repetible. Se usa extensión simétrica y una longitud impar para ejercitar la posible muestra adicional de síntesis. Tras recortar a la longitud de entrada, se comprueba el error con tolerancias absoluta y relativa.

In [ ]:
import numpy as np
import pywt

# Semilla, familia, modo y longitud exclusivamente demostrativos.
samples = np.random.default_rng(20261012).standard_normal(511)
coefficients = pywt.wavedec(samples, "db2", mode="symmetric", level=3)
recovered = pywt.waverec(coefficients, "db2", mode="symmetric")[: samples.size]
np.testing.assert_allclose(recovered, samples, rtol=1e-11, atol=1e-11)
print("Reconstrucción DWT correcta; error máximo:", np.max(np.abs(samples - recovered)))

La comprobación completa se ejecuta desde la raíz con `python scripts/check_environment.py`. Genera el informe, la tabla y la figura sintética. No se extraen conclusiones sobre ventajas de los métodos a partir de estas pruebas.